# FLAMINGO integrated lightcone maps — yang26 / lightcone0

This notebook loads and visualizes **integrated HEALPix maps** from the
[FLAMINGO](https://arxiv.org/abs/2604.24324) **L2p8_m9** simulation, using the
mock CMB secondary-anisotropy products described in
**Yang et al. (2026, MNRAS 548, 1–26; [arXiv:2512.09891](https://doi.org/10.48550/arxiv.2512.09891))**.

## Data location

Local path (downloaded from COSMA DataWeb):

```
/home/ext_andyxlcnb_gmail_com/cosmology_data/flamingo/L2p8_m9/integrated_maps/yang26/lightcone0_shells/
```

Remote source:
[COSMA viewer — yang26 / lightcone0_shells](https://dataweb.cosma.dur.ac.uk:8443/flamingo/viewer.html?path=FLAMINGO%2FL2p8_m9%2FL2p8_m9%2Fintegrated_maps%2Fyang26%2Flightcone0_shells)

## Simulation and lightcone context

| Item | Value |
|------|-------|
| Simulation | **L2p8_m9** — fiducial FLAMINGO hydrodynamical run |
| Box size | 2.8 cGpc (`L2p8`) |
| Mass resolution | m9 (mean baryon particle mass $\approx 1.07\times10^{9}\,M_\odot$) |
| Product line | **yang26** — integrated mock maps from Yang et al. (2026) |
| Lightcone | **lightcone0** — one of **eight** independent lightcone outputs for L2p8 |
| Redshift coverage | Shell maps integrated/stacked to **$z = 4.5$** (fiducial 2.8 cGpc run) |
| HEALPix resolution | $N_\mathrm{side} = 4096$ ($\approx 50''$ pixels; $\sim 1.5$ GB per file in float64) |

### Lightcone construction (Yang et al. 2026, §3.1)

Each observer's past lightcone is split into concentric spherical shells.
Particles crossing the lightcone are assigned to a shell and accumulated onto
HEALPix maps. Shell widths are $\Delta z = 0.05$ for $z < 3$ and
$\Delta z = 0.25$ at higher redshift.

### Why shell rotations (`*_rot_*`) and `same_rot`?

Shell widths can exceed the periodic box size, so photons may traverse repeated
copies of the same structure. To suppress spurious line-of-sight repetitions,
shells are **randomly rotated on the sky every box-length interval** (Yang et
al. 2026, §3.1). The suffix **`same_rot`** indicates that the **same rotation**
was applied to every observable in this set, preserving cross-correlations
(e.g. CIB × tSZ, $\kappa_\mathrm{CMB}$ × CIB).

### Lensing of secondary maps (Yang et al. 2026, §3.7)

Secondary maps are **lensed shell-by-shell** with [pixell](https://github.com/simonsobs/pixell),
using the cumulative CMB convergence map $\kappa$ integrated up to each shell.
This produces the `lensed_` prefix on most files. Lensing modifies small-scale
power by only $\sim 1$–$2\%$ but may matter for future high-precision analyses.

## File format

Each `.hdf5` file contains one dataset:

- **`data`**: 1D HEALPix map, length $N_\mathrm{pix} = 12 N_\mathrm{side}^2 = 201{,}326{,}592$

## Map inventory (8 files, ~12.1 GB total)

| File | Physical quantity (Yang et al. 2026) |
|------|--------------------------------------|
| `CMB_lensing_rot_same_rot.hdf5` | CMB lensing convergence $\kappa_\mathrm{CMB}$ from **total-matter** overdensity shells, integrated to $z=4.5$ with the CMB lensing kernel (Born approximation; §3.2) |
| `lensed_tSZ_rot_same_rot.hdf5` | Lensed Compton-$y$ (tSZ) map; star-forming and recently AGN-heated gas excluded (§3.3) |
| `lensed_kSZ_rot_same_rot.hdf5` | Lensed kinetic SZ: stored field corresponds to $T_\mathrm{kSZ}/T_\mathrm{CMB} = -b$ with $T_\mathrm{CMB}=2.73$ K (§3.3) |
| `lensed_DM_rot_same_rot.hdf5` | Lensed, line-of-sight integrated **dark matter projected mass** per pixel |
| `lensed_CIB_rot_BANDPASS_F217_...` | Lensed CIB at 217 GHz (Planck bandpass) |
| `lensed_CIB_rot_BANDPASS_F353_...` | Lensed CIB at 353 GHz |
| `lensed_CIB_rot_BANDPASS_F545_...` | Lensed CIB at 545 GHz |
| `lensed_CIB_rot_BANDPASS_F857_...` | Lensed CIB at 857 GHz |

CIB maps (§3.5) convert SFR lightcone outputs to IR luminosity (Kennicutt 1998),
apply a greybody SED with **three free parameters** $\theta = [\beta_d, T_0, \alpha]$
fit to Planck CIB auto-power spectra from Lenz et al. (2019), and convolve with
Planck detector bandpasses.

## Notebook strategy

1. Load each map from HDF5 one at a time (~1.6 GB RAM per map).
2. Downgrade to $N_\mathrm{side}=512$ (or 256 for overview panels) for faster Mollweide rendering via `healpy.ud_grade`.
3. Use percentile-based colour limits where dynamic range is large (CIB, DM mass).

> **Kernel**: use the `tpu_rl` environment (`/home/ext_andyxlcnb_gmail_com/envs/tpu_rl`), which has `h5py`, `healpy`, and `matplotlib`.

In [ ]:
from pathlib import Path

import h5py
import healpy as hp
import matplotlib.pyplot as plt
import numpy as np

# --- paths and HEALPix settings ---
DATA_DIR = Path(
    "/home/ext_andyxlcnb_gmail_com/cosmology_data/flamingo"
    "/L2p8_m9/integrated_maps/yang26/lightcone0_shells"
)
NSIDE_FULL = 4096          # native Nside in the HDF5 files (Yang et al. 2026, §3.1)
NSIDE_PLOT = 512           # downgrade for responsive Mollweide rendering
Z_MAX = 4.5                # integration limit for L2p8_m9 mock maps (Yang et al. 2026)

assert DATA_DIR.is_dir(), f"Data directory not found: {DATA_DIR}"
print(f"Data directory: {DATA_DIR}")
print(f"Files found: {len(list(DATA_DIR.glob('*.hdf5')))}")
print(f"Expected Nside: {NSIDE_FULL}, integration redshift: z = {Z_MAX}")

In [ ]:
def load_healpix_map(hdf5_path: Path) -> np.ndarray:
    """Load the 'data' dataset from a FLAMINGO integrated-map HDF5 file."""
    with h5py.File(hdf5_path, "r") as f:
        m = f["data"][:]
    expected_npix = 12 * NSIDE_FULL ** 2
    if m.size != expected_npix:
        raise ValueError(f"{hdf5_path.name}: expected {expected_npix} pixels, got {m.size}")
    return m.astype(np.float64)


def summarize_map(m: np.ndarray, name: str) -> None:
    """Print basic pixel statistics."""
    finite = np.isfinite(m)
    print(f"\n{name}")
    print(f"  pixels : {m.size:,}")
    print(f"  min    : {np.nanmin(m):.6e}")
    print(f"  max    : {np.nanmax(m):.6e}")
    print(f"  mean   : {np.nanmean(m):.6e}")
    print(f"  std    : {np.nanstd(m):.6e}")
    print(f"  finite : {finite.mean() * 100:.2f}%")


def plot_mollweide(
    m: np.ndarray,
    title: str,
    unit: str = "",
    cmap: str = "RdBu_r",
    log_scale: bool = False,
    pct_limits: tuple[float, float] = (2, 98),
    nside_plot: int = NSIDE_PLOT,
) -> None:
    """Downgrade and render a full-sky Mollweide map."""
    if nside_plot < NSIDE_FULL:
        m_plot = hp.ud_grade(m, nside_plot)
    else:
        m_plot = m.copy()

    if log_scale:
        # shift to positive values for log display
        offset = np.nanpercentile(m_plot, pct_limits[0])
        m_plot = np.where(m_plot > 0, m_plot, np.nan)
        vmin, vmax = np.nanpercentile(m_plot, pct_limits)
        hp.mollview(
            m_plot,
            title=title,
            unit=unit,
            cmap=cmap,
            norm="log",
            min=vmin,
            max=vmax,
            hold=True,
        )
    else:
        vmin, vmax = np.nanpercentile(m_plot, pct_limits)
        # symmetric limits for signed fields (SZ, kappa)
        if np.nanmin(m_plot) < 0 and np.nanmax(m_plot) > 0:
            lim = max(abs(vmin), abs(vmax))
            vmin, vmax = -lim, lim
        hp.mollview(
            m_plot,
            title=title,
            unit=unit,
            cmap=cmap,
            min=vmin,
            max=vmax,
            hold=True,
        )
    hp.graticule(dmer=30, dpar=30, verbose=False)
    plt.show()

## 1. CMB lensing convergence ($\kappa_\mathrm{CMB}$)

Following Yang et al. (2026, §3.2), we first compute the 2D projected **total-matter**
overdensity $\delta(\chi, \theta)$ in each lightcone shell, then integrate along the line of
sight to $z = 4.5$ weighted by the CMB lensing kernel $W_\kappa^\mathrm{CMB}(\chi)$
(Born approximation). The result is the convergence field $\kappa_\mathrm{CMB}(\theta)$ used
to characterize gravitational deflection of CMB photons by large-scale structure.

In [ ]:
kappa = load_healpix_map(DATA_DIR / "CMB_lensing_rot_same_rot.hdf5")
summarize_map(kappa, "CMB lensing κ")
plot_mollweide(kappa, title="CMB lensing convergence κ (yang26, lightcone0)", unit="κ", cmap="PuOr_r")

## 2. Thermal Sunyaev–Zel'dovich (tSZ)

The tSZ effect (Yang et al. 2026, §3.3) arises from inverse Compton scattering of CMB
photons off hot electrons. The map stores the dimensionless **Compton-$y$** parameter,
integrated over lightcone shells to $z = 4.5$. Contributions from star-forming gas and
from gas recently heated by AGN feedback are excluded. Lensing has been applied
shell-by-shell (§3.7). Brightest features trace massive clusters and groups.

In [ ]:
tsz = load_healpix_map(DATA_DIR / "lensed_tSZ_rot_same_rot.hdf5")
summarize_map(tsz, "lensed tSZ (Compton y)")
plot_mollweide(
    tsz,
    title="Lensed tSZ — Compton y (yang26, lightcone0)",
    unit="y",
    cmap="hot_r",
    pct_limits=(0.5, 99.5),
)

## 3. Kinetic Sunyaev–Zel'dovich (kSZ)

The kSZ effect probes line-of-sight peculiar velocities of free electrons (Yang et al.
2026, §3.3). FLAMINGO stores the dimensionless Doppler parameter $b$, related to the
temperature fluctuation by $T_\mathrm{kSZ}/T_\mathrm{CMB} = -b$ with
$T_\mathrm{CMB} = 2.73\,\mathrm{K}$. The same gas-particle exclusions as for tSZ apply.
Values are small compared to tSZ and show both large-scale dipolar and small-scale structure.

In [ ]:
ksz = load_healpix_map(DATA_DIR / "lensed_kSZ_rot_same_rot.hdf5")
summarize_map(ksz, "lensed kSZ (T_kSZ/T_CMB = -b)")
plot_mollweide(
    ksz,
    title="Lensed kSZ — T_kSZ/T_CMB (yang26, lightcone0)",
    unit="T_kSZ/T_CMB",
    cmap="RdBu_r",
)

## 4. Dark matter projected mass

The `lensed_DM` map stores the **line-of-sight integrated dark matter mass per HEALPix
pixel**, with gravitational lensing applied shell-by-shell. This traces the collisionless
matter distribution and complements the gas-based tSZ/kSZ and SFR-based CIB maps.
Values are in simulation mass units accumulated per pixel (hence the large dynamic range).

In [ ]:
dm = load_healpix_map(DATA_DIR / "lensed_DM_rot_same_rot.hdf5")
summarize_map(dm, "lensed DM projected mass")
plot_mollweide(
    dm,
    title="Lensed dark-matter projected mass (yang26, lightcone0)",
    unit="M_sim / pixel",
    cmap="viridis",
    log_scale=True,
    pct_limits=(5, 99),
)

## 5. Cosmic Infrared Background (CIB) — four Planck bands

CIB maps follow the **three-parameter model** of Yang et al. (2026, §3.5.1). Starting from
FLAMINGO SFR lightcone shells, the mapping to observed intensity is:

**1. SFR → bolometric IR luminosity** (Kennicutt 1998; Chabrier IMF):

$$
\frac{L_{\mathrm{bol,IR}}}{1\times 10^{10}\,L_\odot}
=
\frac{\mathrm{SFR}}{1\,M_\odot\,\mathrm{yr}^{-1}}
\qquad (11)
$$

**2. Dust temperature evolution** (power law in redshift):

$$
T_{\mathrm{dust}}(z) = T_0\,(1+z)^{\alpha}
$$

**3. Greybody SED** (Planck Collaboration XV 2016a). For the frequencies and redshifts
used here ($z\le 4.5$, $\nu < 1000\,\mathrm{GHz}$), the high-frequency power-law cut-off is
not needed, so

$$
\Theta(\nu, T_{\mathrm{dust}}, z)
=
\left[\exp\!\left(\frac{h\nu}{k_B T_{\mathrm{dust}}(z)}\right)-1\right]^{-1}
\nu^{\beta_d+3}
\qquad (12)
$$

**4. Specific luminosity** at rest-frame frequency $\nu$:

$$
L_{\nu,\mathrm{IR}}
=
L_{\mathrm{bol,IR}}(\mathrm{SFR})\,
\frac{\Theta(\nu, T_{\mathrm{dust}})}{\displaystyle\int\mathrm{d}\nu'\,\Theta(\nu', T_{\mathrm{dust}})}
\qquad (13)
$$

**5. Observed flux** (and Planck bandpass convolution):

$$
S_{\nu,\mathrm{IR}}
=
\frac{L_{\nu(1+z),\mathrm{IR}}}{4\pi\,\chi^2\,(1+z)}
\qquad (14)
$$

The three free parameters $\boldsymbol{\theta}=[\beta_d,\,T_0,\,\alpha]$ are fit to the auto-power spectra
of Lenz et al. (2019) at 353, 545, and 857 GHz; the same SED is then evaluated (and
convolved with each Planck HFI bandpass) at **217, 353, 545, and 857 GHz**. Shells are
stacked to $z=4.5$ and lensed shell-by-shell (§3.7).

In [ ]:
cib_bands = [
    (217, "lensed_CIB_rot_BANDPASS_F217_three_params_same_rot.hdf5"),
    (353, "lensed_CIB_rot_BANDPASS_F353_three_params_same_rot.hdf5"),
    (545, "lensed_CIB_rot_BANDPASS_F545_three_params_same_rot.hdf5"),
    (857, "lensed_CIB_rot_BANDPASS_F857_three_params_same_rot.hdf5"),
]

for i, (freq, fname) in enumerate(cib_bands, start=1):
    m = load_healpix_map(DATA_DIR / fname)
    summarize_map(m, f"CIB {freq} GHz")
    m_plot = hp.ud_grade(m, NSIDE_PLOT)
    m_pos = np.where(m_plot > 0, m_plot, np.nan)
    vmin, vmax = np.nanpercentile(m_pos, [5, 99])
    hp.mollview(
        m_plot,
        title=f"Lensed CIB @ {freq} GHz",
        sub=(2, 2, i),
        cmap="Greys",
        norm="log",
        min=vmin,
        max=vmax,
        hold=True,
    )
    hp.graticule(dmer=30, dpar=30, verbose=False)

plt.suptitle("CIB integrated maps — yang26 / lightcone0 (all bands)", y=1.02, fontsize=14)
plt.show()

## 6. Overview — all maps at a glance

Quick-look panel of every map type (downgraded to $N_\mathrm{side}=256$ for speed).

In [ ]:
MAP_CATALOG = [
    ("CMB_lensing_rot_same_rot.hdf5", "CMB lensing κ", "PuOr_r", False),
    ("lensed_tSZ_rot_same_rot.hdf5", "tSZ (y)", "hot_r", False),
    ("lensed_kSZ_rot_same_rot.hdf5", "kSZ (T/T_CMB)", "RdBu_r", False),
    ("lensed_DM_rot_same_rot.hdf5", "DM projected mass", "viridis", True),
    ("lensed_CIB_rot_BANDPASS_F353_three_params_same_rot.hdf5", "CIB 353 GHz", "Greys", True),
]

NSIDE_THUMB = 256
fig = plt.figure(figsize=(16, 9))

for i, (fname, label, cmap, use_log) in enumerate(MAP_CATALOG, start=1):
    m = load_healpix_map(DATA_DIR / fname)
    m_plot = hp.ud_grade(m, NSIDE_THUMB)
    if use_log:
        m_pos = np.where(m_plot > 0, m_plot, np.nan)
        vmin, vmax = np.nanpercentile(m_pos, [5, 99])
        hp.mollview(
            m_plot, title=label, sub=(2, 3, i), cmap=cmap,
            norm="log", min=vmin, max=vmax, hold=True,
        )
    else:
        vmin, vmax = np.nanpercentile(m_plot, [2, 98])
        if np.nanmin(m_plot) < 0:
            lim = max(abs(vmin), abs(vmax))
            vmin, vmax = -lim, lim
        hp.mollview(
            m_plot, title=label, sub=(2, 3, i), cmap=cmap,
            min=vmin, max=vmax, hold=True,
        )

plt.suptitle(
    f"FLAMINGO L2p8_m9 integrated maps — yang26 / lightcone0 (Nside={NSIDE_THUMB})",
    fontsize=14, y=1.02,
)
plt.tight_layout()
plt.show()

## References

1. **Yang et al. (2026)** — *Self-consistent secondary cosmic microwave background anisotropies and extragalactic foregrounds in the flamingo simulations*, MNRAS 548, 1–26. [doi:10.1093/mnras/stag625](https://doi.org/10.1093/mnras/stag625) · [arXiv:2512.09891](https://doi.org/10.48550/arxiv.2512.09891)
2. **Schaye et al. (2026)** — *The FLAMINGO simulations data release*, arXiv:2604.24324
3. **Schaye et al. (2023)** — *The FLAMINGO project*, MNRAS 526, 4978
4. **Lenz et al. (2019)** — Planck CIB auto-power spectra used to calibrate the three-parameter SED model
5. [FLAMINGO DataWeb documentation](https://dataweb.cosma.dur.ac.uk:8443/flamingo/)

### Next steps

- Cross-correlate tSZ with $\kappa_\mathrm{CMB}$ using these `same_rot` maps (Yang et al. 2026, §4)
- Compare individual redshift shells vs. integrated maps
- Download other lightcones (`lightcone1`–`7`; eight independent realizations for L2p8)
- Use $N_\mathrm{side}=16384$ maps for cluster-scale studies (much larger files)